In [1]:
import os
import random
import time
from enum import IntEnum
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =====================================================================
# PART 1: FOUNDATION & ENUMS
# =====================================================================
class Action(IntEnum):
    NORTH = 0
    SOUTH = 1
    EAST = 2
    WEST = 3

class MissionPhase(IntEnum):
    GOING_TO_A = 0  # Pickup phase (A -> B direction vector: returning to A)
    GOING_TO_B = 1  # Dropoff phase (A -> B direction vector: carrying item to B)

MOVE_OFFSETS = {
    Action.NORTH: (-1, 0),
    Action.SOUTH: (1, 0),
    Action.EAST:  (0, 1),
    Action.WEST:  (0, -1)
}

# =====================================================================
# PART 2: ENVIRONMENT DESIGN
# =====================================================================
class MultiAgentGridWorld:
    """
    5x5 Grid World with 4 shuttle agents.
    Handles continuous shuttle execution, wall collisions, and direction-based head-on collisions.
    """
    def __init__(self, grid_size=5, num_agents=4, loc_A=(0, 0), loc_B=(4, 4)):
        self.grid_size = grid_size
        self.num_agents = num_agents
        self.loc_A = loc_A
        self.loc_B = loc_B
        
        # State tracking per agent
        self.agent_positions = [list(loc_A) for _ in range(num_agents)]
        self.agent_phases = [MissionPhase.GOING_TO_A for _ in range(num_agents)]
        self.agent_carrying = [False for _ in range(num_agents)]
        self.deliveries_completed = [0 for _ in range(num_agents)]
        
        self.total_steps = 0
        self.total_collisions = 0
        
        self.reset()

    def reset(self, random_starts=False):
        self.agent_positions = []
        for i in range(self.num_agents):
            if random_starts:
                r = random.randint(0, self.grid_size - 1)
                c = random.randint(0, self.grid_size - 1)
                self.agent_positions.append([r, c])
            else:
                self.agent_positions.append(list(self.loc_A))
                
        self.agent_phases = [MissionPhase.GOING_TO_A] * self.num_agents
        self.agent_carrying = [False] * self.num_agents
        self.deliveries_completed = [0] * self.num_agents
        return [self.get_observation(i) for i in range(self.num_agents)]

    def get_observation(self, agent_id):
        """
        State representation (13 dimensions):
        - One-hot agent index (4)
        - Agent row, col normalized (2)
        - Loc A row, col normalized (2)
        - Loc B row, col normalized (2)
        - Carrying flag (1)
        - Target destination row, col normalized (2)
        """
        obs = np.zeros(13, dtype=np.float32)
        obs[agent_id] = 1.0  # One-hot encoding
        
        r, c = self.agent_positions[agent_id]
        obs[4] = r / (self.grid_size - 1)
        obs[5] = c / (self.grid_size - 1)
        
        obs[6] = self.loc_A[0] / (self.grid_size - 1)
        obs[7] = self.loc_A[1] / (self.grid_size - 1)
        
        obs[8] = self.loc_B[0] / (self.grid_size - 1)
        obs[9] = self.loc_B[1] / (self.grid_size - 1)
        
        obs[10] = 1.0 if self.agent_carrying[agent_id] else 0.0
        
        target = self.loc_B if self.agent_carrying[agent_id] else self.loc_A
        obs[11] = target[0] / (self.grid_size - 1)
        obs[12] = target[1] / (self.grid_size - 1)
        
        return obs

    def step(self, actions):
        """
        Executes sequential movement in random update order.
        Returns: next_observations, rewards, done, info
        """
        agent_order = list(range(self.num_agents))
        random.shuffle(agent_order)
        
        rewards = [0.0] * self.num_agents
        head_on_collision_occurred = False
        
        for idx in agent_order:
            act = actions[idx]
            dr, dc = MOVE_OFFSETS[act]
            curr_r, curr_c = self.agent_positions[idx]
            new_r = curr_r + dr
            new_c = curr_c + dc
            
            # Step penalty to encourage efficiency
            rewards[idx] -= 0.1
            
            # Wall Collision handling
            hit_wall = False
            if not (0 <= new_r < self.grid_size and 0 <= new_c < self.grid_size):
                hit_wall = True
                new_r, new_c = curr_r, curr_c  # Stay in place
                rewards[idx] -= 1.0  # Wall penalty
            
            self.agent_positions[idx] = [new_r, new_c]
            
            # Distance-based shaping reward
            target = self.loc_B if self.agent_carrying[idx] else self.loc_A
            old_dist = abs(curr_r - target[0]) + abs(curr_c - target[1])
            new_dist = abs(new_r - target[0]) + abs(new_c - target[1])
            if not hit_wall:
                if new_dist < old_dist:
                    rewards[idx] += 0.2
                else:
                    rewards[idx] -= 0.2

            # Mission State Machine & Shuttle Logic
            curr_pos = (new_r, new_c)
            if curr_pos == self.loc_A and not self.agent_carrying[idx]:
                self.agent_carrying[idx] = True
                self.agent_phases[idx] = MissionPhase.GOING_TO_B
                rewards[idx] += 5.0  # Successful pickup
                
            elif curr_pos == self.loc_B and self.agent_carrying[idx]:
                self.agent_carrying[idx] = False
                self.agent_phases[idx] = MissionPhase.GOING_TO_A
                self.deliveries_completed[idx] += 1
                rewards[idx] += 10.0  # Successful dropoff

        # Check for Head-On Collisions after all agents moved
        collision_count = self._evaluate_collisions()
        if collision_count > 0:
            head_on_collision_occurred = True
            self.total_collisions += collision_count
            for i in range(self.num_agents):
                rewards[i] -= 15.0  # Heavy collision penalty for all involved/nearby

        self.total_steps += self.num_agents
        next_obs = [self.get_observation(i) for i in range(self.num_agents)]
        
        info = {
            "head_on_collisions": collision_count,
            "deliveries": sum(self.deliveries_completed)
        }
        
        return next_obs, rewards, head_on_collision_occurred, info

    def _evaluate_collisions(self):
        """
        Head-on collision: Agents sharing the same cell outside A and B with OPPOSITE mission phases.
        """
        collisions = 0
        cell_occupancy = {}
        
        for i in range(self.num_agents):
            pos = tuple(self.agent_positions[i])
            if pos == self.loc_A or pos == self.loc_B:
                continue  # Disregard collisions at pickup and delivery sites
                
            phase = self.agent_phases[i]
            if pos not in cell_occupancy:
                cell_occupancy[pos] = set()
            cell_occupancy[pos].add(phase)
            
        for pos, phases in cell_occupancy.items():
            # If both GOING_TO_A and GOING_TO_B exist in the same cell -> Head-on Collision
            if len(phases) > 1:
                collisions += 1
                
        return collisions

# =====================================================================
# PART 3 & 4: DEEP Q-NETWORK & REPLAY BUFFER
# =====================================================================
class QNetwork(nn.Module):
    def __init__(self, state_dim=13, action_dim=4):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

class ReplayBuffer:
    def __init__(self, capacity=200000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (
            torch.FloatTensor(np.array(state)),
            torch.LongTensor(action),
            torch.FloatTensor(reward),
            torch.FloatTensor(np.array(next_state)),
            torch.FloatTensor(done)
        )

    def __len__(self):
        return len(self.buffer)

# =====================================================================
# PART 5 & 6: TRAINING PIPELINE
# =====================================================================
class DQNAgentSystem:
    def __init__(self, state_dim=13, action_dim=4, lr=1e-3, gamma=0.98):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q_net = QNetwork(state_dim, action_dim).to(self.device)
        self.target_net = QNetwork(state_dim, action_dim).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer()
        
        self.gamma = gamma
        self.batch_size = 64
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.999995

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.q_net(state_t)
        return torch.argmax(q_values, dim=1).item()

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
            
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        
        states = states.to(self.device)
        actions = actions.unsqueeze(1).to(self.device)
        rewards = rewards.unsqueeze(1).to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.unsqueeze(1).to(self.device)

        # Current Q-values
        curr_q = self.q_net(states).gather(1, actions)
        
        # Double DQN target computation
        with torch.no_grad():
            next_actions = self.q_net(next_states).argmax(dim=1, keepdim=True)
            max_next_q = self.target_net(next_states).gather(1, next_actions)
            target_q = rewards + (1 - dones) * self.gamma * max_next_q

        loss = nn.MSELoss()(curr_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=1.0)
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        return loss.item()

    def update_target_network(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

def train_system():
    print("=========================================================")
    print("Starting Multi-Agent DQN Training Environment")
    print("Budgets: Steps <= 1,500,000 | Collisions <= 4,000 | Walltime <= 10m")
    print("=========================================================")
    
    env = MultiAgentGridWorld()
    trainer = DQNAgentSystem()
    
    start_time = time.time()
    max_walltime_sec = 2 * 60  # 10 minutes limit
    max_step_budget = 1500
    max_collision_budget = 400
    
    obs = env.reset(random_starts=True)
    step_count = 0
    episode_collisions = 0
    
    sync_target_freq = 100
    
    while True:
        # Check training budgets
        elapsed_time = time.time() - start_time
        if elapsed_time >= max_walltime_sec:
            print(f"\n[STOP] Walltime budget reached ({elapsed_time/60:.2f} min).")
            break
        if env.total_steps >= max_step_budget:
            print(f"\n[STOP] Total step budget reached ({env.total_steps}).")
            break
        if env.total_collisions >= max_collision_budget:
            print(f"\n[STOP] Collision budget reached ({env.total_collisions}).")
            break

        # Action selection for all 4 agents
        actions = [trainer.select_action(obs[i]) for i in range(env.num_agents)]
        
        # Environment transition
        next_obs, rewards, collision, info = env.step(actions)
        
        # Store individual transitions into experience replay
        for i in range(env.num_agents):
            trainer.buffer.push(obs[i], actions[i], rewards[i], next_obs[i], False)
            
        obs = next_obs
        step_count += 1
        
        # Model update
        trainer.train_step()
        
        if step_count % sync_target_freq == 0:
            trainer.update_target_network()

        # Logging periodic progress
        if step_count % 25000 == 0:
            print(f"Steps: {env.total_steps:7d} | Collisions: {env.total_collisions:4d} | "
                  f"Epsilon: {trainer.epsilon:.3f} | Walltime: {elapsed_time/60:.2f}m")

    print("\nTraining completed successfully!")
    print(f"Total Steps Executed: {env.total_steps}")
    print(f"Total Head-On Collisions: {env.total_collisions}")
    print(f"Total Training Time: {(time.time() - start_time)/60:.2f} minutes")
    
    return env, trainer

# =====================================================================
# PART 7: EVALUATION & SCENARIO TESTING
# =====================================================================
def evaluate_performance(trainer, num_scenarios=200):
    """
    Evaluates agents on single deliveries starting at B within at most 25 steps, collision-free.
    A single delivery is defined as: B -> A (Pickup) -> B (Deliver).
    """
    print("\n=========================================================")
    print(f"Running Performance Evaluation ({num_scenarios} Test Scenarios)")
    print("=========================================================")
    
    env = MultiAgentGridWorld()
    trainer.q_net.eval()
    
    successful_deliveries = 0
    total_eval_collisions = 0
    step_records = []

    for scenario in range(num_scenarios):
        # Start all agents at Location B
        env.agent_positions = [list(env.loc_B) for _ in range(env.num_agents)]
        env.agent_phases = [MissionPhase.GOING_TO_A] * env.num_agents
        env.agent_carrying = [False] * env.num_agents
        env.deliveries_completed = [0] * env.num_agents
        
        obs = [env.get_observation(i) for i in range(env.num_agents)]
        scenario_collision = False
        steps_taken = 0
        
        for step in range(25):
            steps_taken += 1
            actions = []
            for i in range(env.num_agents):
                state_t = torch.FloatTensor(obs[i]).unsqueeze(0).to(trainer.device)
                with torch.no_grad():
                    act = torch.argmax(trainer.q_net(state_t), dim=1).item()
                actions.append(act)
                
            next_obs, _, collision, info = env.step(actions)
            obs = next_obs
            
            if collision:
                scenario_collision = True
                total_eval_collisions += 1

            # Check if all agents completed at least one delivery
            if all(d >= 1 for d in env.deliveries_completed):
                break

        if not scenario_collision and all(d >= 1 for d in env.deliveries_completed) and steps_taken <= 25:
            successful_deliveries += 1
            
        step_records.append(steps_taken)

    success_rate = (successful_deliveries / num_scenarios) * 100
    print(f"\nEvaluation Results:")
    print(f"Success Rate (<25 steps, collision-free): {success_rate:.2f}%")
    print(f"Total Collisions in Eval: {total_eval_collisions}")
    print(f"Average Steps per Delivery: {np.mean(step_records):.2f}")
    
    return success_rate, total_eval_collisions

# =====================================================================
# PART 8: VISUALIZATION
# =====================================================================
def visualize_grid(env):
    """
    Renders 5x5 Grid showing agent positions without visual overlap.
    """
    grid = np.full((env.grid_size, env.grid_size), ".", dtype=object)
    grid[env.loc_A[0], env.loc_A[1]] = "A"
    grid[env.loc_B[0], env.loc_B[1]] = "B"
    
    pos_map = {}
    for idx, pos in enumerate(env.agent_positions):
        t_pos = tuple(pos)
        if t_pos not in pos_map:
            pos_map[t_pos] = []
        pos_map[t_pos].append(f"Ag{idx}")

    print("\n--- Current Grid World View ---")
    for r in range(env.grid_size):
        row_str = []
        for c in range(env.grid_size):
            cell_key = (r, c)
            if cell_key in pos_map:
                row_str.append(f"[{','.join(pos_map[cell_key])}]")
            elif (r, c) == env.loc_A:
                row_str.append("[  A  ]")
            elif (r, c) == env.loc_B:
                row_str.append("[  B  ]")
            else:
                row_str.append("[  .  ]")
        print(" ".join(row_str))
    print("--------------------------------\n")

# =====================================================================
# MAIN EXECUTION ENTRYPOINT
# =====================================================================
if __name__ == "__main__":
    # Execute Training
    env, trainer = train_system()
    
    # Render final state layout
    visualize_grid(env)
    
    # Run evaluation checks
    success_rate, eval_collisions = evaluate_performance(trainer)
    
    # Scaling factor alpha calculation
    B_points = 2 if success_rate > 95 and env.total_collisions < 500 else (1 if success_rate > 85 and env.total_collisions < 1000 else 0)
    C_cost = 0  # Zero cost strategy chosen
    alpha = 1 - (33 / 200) * max(0, C_cost - B_points)
    
    print("\n=========================================================")
    print("FINAL PERFORMANCE METRICS SUMMARY")
    print("=========================================================")
    print(f"Cost Purchased (C): {C_cost}")
    print(f"Performance Points (B): {B_points}")
    print(f"Grade Scaling Factor (alpha): {alpha:.3f}")
    print("=========================================================")

Starting Multi-Agent DQN Training Environment
Budgets: Steps <= 1,500,000 | Collisions <= 4,000 | Walltime <= 10m
Steps:  100000 | Collisions: 2457 | Epsilon: 0.883 | Walltime: 0.97m

[STOP] Collision budget reached (4000).

Training completed successfully!
Total Steps Executed: 160568
Total Head-On Collisions: 4000
Total Training Time: 1.70 minutes

--- Current Grid World View ---
[  A  ] [  .  ] [  .  ] [  .  ] [  .  ]
[Ag1] [Ag0] [  .  ] [  .  ] [  .  ]
[  .  ] [  .  ] [  .  ] [  .  ] [Ag2,Ag3]
[  .  ] [  .  ] [  .  ] [  .  ] [  .  ]
[  .  ] [  .  ] [  .  ] [  .  ] [  B  ]
--------------------------------


Running Performance Evaluation (200 Test Scenarios)

Evaluation Results:
Success Rate (<25 steps, collision-free): 100.00%
Total Collisions in Eval: 0
Average Steps per Delivery: 16.00

FINAL PERFORMANCE METRICS SUMMARY
Cost Purchased (C): 0
Performance Points (B): 0
Grade Scaling Factor (alpha): 1.000


In [2]:
import time
from IPython.display import clear_output
import matplotlib.pyplot as plt

def render_grid_jupyter(env, step_delay=0.3, step_num=0):
    """
    Jupyter Notebook-e live graphical animation dekhabor jonno updated visualizer.
    """
    clear_output(wait=True)  # Purono frame clear kore dynamic effect ane
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-0.5, env.grid_size - 0.5)
    ax.set_ylim(-0.5, env.grid_size - 0.5)
    ax.set_xticks(range(env.grid_size))
    ax.set_yticks(range(env.grid_size))
    ax.grid(True, which='both', color='black', linestyle='-', linewidth=1)
    ax.invert_yaxis()  # Grid upper-left (0,0) korar jonno
    
    # Pickup A and Dropoff B locations
    ax.text(env.loc_A[1], env.loc_A[0], 'A\n(Pickup)', fontsize=12, ha='center', va='center', 
            bbox=dict(boxstyle='square,pad=0.5', facecolor='lightgreen', alpha=0.7))
    ax.text(env.loc_B[1], env.loc_B[0], 'B\n(Dropoff)', fontsize=12, ha='center', va='center', 
            bbox=dict(boxstyle='square,pad=0.5', facecolor='coral', alpha=0.7))
    
    # Render Agents with offset if on same cell
    agent_colors = ['blue', 'purple', 'darkgoldenrod', 'crimson']
    cell_counts = {}
    
    for i, pos in enumerate(env.agent_positions):
        r, c = pos
        cell = (r, c)
        cell_counts[cell] = cell_counts.get(cell, 0) + 1
        offset = (cell_counts[cell] - 1) * 0.15  # Visual overlap prevent korte
        
        status = "B" if env.agent_carrying[i] else "A"
        ax.plot(c + offset - 0.05, r, marker='o', markersize=18, color=agent_colors[i])
        ax.text(c + offset - 0.05, r, f"Ag{i}\n({status})", color='white', fontsize=7, 
                ha='center', va='center', weight='bold')

    plt.title(f"Multi-Agent Grid World | Step: {step_num}", fontsize=14)
    plt.show()
    
    time.sleep(step_delay)  # Agent movement slow-motion dekhabor jonno

In [1]:
def run_live_demo(env, trainer, steps=25):
    """
    Trained agents-er live shuttle movement dekhabe.
    """
    # Test scenario start at B location
    env.agent_positions = [list(env.loc_B) for _ in range(env.num_agents)]
    env.agent_phases = [MissionPhase.GOING_TO_A] * env.num_agents
    env.agent_carrying = [False] * env.num_agents
    
    obs = [env.get_observation(i) for i in range(env.num_agents)]
    
    for s in range(1, steps + 1):
        # Render current frame
        render_grid_jupyter(env, step_delay=0.4, step_num=s)
        
        # Action selection from Q-network
        actions = []
        for i in range(env.num_agents):
            state_t = torch.FloatTensor(obs[i]).unsqueeze(0).to(trainer.device)
            with torch.no_grad():
                act = torch.argmax(trainer.q_net(state_t), dim=1).item()
            actions.append(act)
            
        # Move agents
        next_obs, _, collision, info = env.step(actions)
        obs = next_obs
        
        if collision:
            print("⚠️ HEAD-ON COLLISION DETECTED!")

# Training-er por live demo run koro:
run_live_demo(env, trainer, steps=25)

NameError: name 'env' is not defined